# CodSoft Task 1 — Titanic Survival Prediction

This project builds a machine learning classification model to predict whether a Titanic passenger survived.

**Dataset:** `Titanic-Dataset.csv`

### Workflow
1. Load and inspect the dataset
2. Exploratory Data Analysis
3. Data preprocessing
4. Feature engineering
5. Train/test split
6. Logistic Regression
7. Random Forest comparison
8. Model evaluation
9. Sample predictions


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


In [ ]:
# Load the dataset
df = pd.read_csv("Titanic-Dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# Basic dataset information
print("Data types and non-null counts:")
df.info()

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
# Descriptive statistics
display(df.describe(include="all").T)


## Exploratory Data Analysis

In [ ]:
# Survival distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Survived")
plt.title("Survival Distribution")
plt.xlabel("Survived (0 = No, 1 = Yes)")
plt.ylabel("Number of Passengers")
plt.show()


In [ ]:
# Survival by passenger class
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="Pclass", hue="Survived")
plt.title("Survival by Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Number of Passengers")
plt.show()


In [ ]:
# Survival by sex
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="Sex", hue="Survived")
plt.title("Survival by Sex")
plt.xlabel("Sex")
plt.ylabel("Number of Passengers")
plt.show()


In [ ]:
# Age distribution by survival
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="Age", hue="Survived", kde=True, bins=30, element="step")
plt.title("Age Distribution by Survival")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()


In [ ]:
# Fare distribution by survival
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Survived", y="Fare")
plt.title("Fare Distribution by Survival")
plt.xlabel("Survived")
plt.ylabel("Fare")
plt.show()


In [ ]:
# Correlation heatmap for numeric variables
plt.figure(figsize=(9, 6))
numeric_df = df.select_dtypes(include=np.number)
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()


## Feature Engineering

In [ ]:
# Create useful features from the original columns
df_model = df.copy()

# Extract passenger title from Name
df_model["Title"] = df_model["Name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip()

# Group uncommon titles into "Rare"
common_titles = ["Mr", "Miss", "Mrs", "Master"]
df_model["Title"] = df_model["Title"].where(
    df_model["Title"].isin(common_titles), "Rare"
)

# Family size
df_model["FamilySize"] = df_model["SibSp"] + df_model["Parch"] + 1

# Whether the passenger was travelling alone
df_model["IsAlone"] = (df_model["FamilySize"] == 1).astype(int)

display(df_model[["Name", "Title", "SibSp", "Parch", "FamilySize", "IsAlone"]].head())


## Selecting Features

`Survived` is the target variable. We remove `PassengerId`, `Name`, `Ticket`, and `Cabin` from the model because they are not being used directly in this implementation.

The remaining features include passenger class, sex, age, family information, fare, embarkation port, and the engineered title/family features.


In [ ]:
target = "Survived"

features = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare",
    "Embarked", "Title", "FamilySize", "IsAlone"
]

X = df_model[features]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set :", X_test.shape)


In [ ]:
# Preprocessing pipelines
numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone"]
categorical_features = ["Sex", "Embarked", "Title"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


## Model 1 — Logistic Regression

In [ ]:
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

logistic_model.fit(X_train, y_train)

y_pred_lr = logistic_model.predict(X_test)
y_prob_lr = logistic_model.predict_proba(X_test)[:, 1]

print("Logistic Regression Classification Report")
print(classification_report(y_test, y_pred_lr))

print("Accuracy :", round(accuracy_score(y_test, y_pred_lr), 4))
print("Precision:", round(precision_score(y_test, y_pred_lr), 4))
print("Recall   :", round(recall_score(y_test, y_pred_lr), 4))
print("F1 Score :", round(f1_score(y_test, y_pred_lr), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_prob_lr), 4))


In [ ]:
# Confusion matrix — Logistic Regression
cm = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Logistic Regression — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
# ROC curve — Logistic Regression
RocCurveDisplay.from_predictions(y_test, y_prob_lr)
plt.title("Logistic Regression — ROC Curve")
plt.show()


## Model 2 — Random Forest

In [ ]:
random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ))
])

random_forest_model.fit(X_train, y_train)

y_pred_rf = random_forest_model.predict(X_test)
y_prob_rf = random_forest_model.predict_proba(X_test)[:, 1]

print("Random Forest Classification Report")
print(classification_report(y_test, y_pred_rf))

print("Accuracy :", round(accuracy_score(y_test, y_pred_rf), 4))
print("Precision:", round(precision_score(y_test, y_pred_rf), 4))
print("Recall   :", round(recall_score(y_test, y_pred_rf), 4))
print("F1 Score :", round(f1_score(y_test, y_pred_rf), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_prob_rf), 4))


In [ ]:
# Compare both models
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf)
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf)
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_rf)
    ]
})

display(results.round(4))


In [ ]:
# Select the model with the higher ROC-AUC
if roc_auc_score(y_test, y_prob_lr) >= roc_auc_score(y_test, y_prob_rf):
    best_model = logistic_model
    best_model_name = "Logistic Regression"
else:
    best_model = random_forest_model
    best_model_name = "Random Forest"

print("Selected model:", best_model_name)


## Example Predictions

In [ ]:
# Predict survival for a few passengers from the test set
sample = X_test.head(5).copy()
sample_predictions = best_model.predict(sample)
sample_probabilities = best_model.predict_proba(sample)[:, 1]

prediction_output = sample.copy()
prediction_output["Predicted_Survival"] = sample_predictions
prediction_output["Survival_Probability"] = sample_probabilities.round(3)

display(prediction_output)


## Conclusion

The Titanic survival problem was treated as a binary classification task. The dataset was inspected, missing values were handled through preprocessing pipelines, categorical features were one-hot encoded, numerical features were scaled, and additional features such as `Title`, `FamilySize`, and `IsAlone` were created.

Two classification algorithms were trained and compared: Logistic Regression and Random Forest. The model with the better ROC-AUC on the held-out test set was selected as the final model.

This completes the core requirements of CodSoft Task 1: Titanic Survival Prediction.
